# TechCare | Data Preprocessing - Phase 2

## Objective

Prepare the synthetic customer-support ticket dataset for loading into MySQL and subsequent analysis in Power BI.

The workflow includes:

- structure and identifier inspection;
- missing-value analysis;
- validation of escalation and ticket status;
- date conversion and verification;
- numeric-variable validation;
- preparation of text fields;
- export for MySQL.

**Input file:** `phase2_support_tickets_raw.csv`  
**Output file:** `phase2_support_tickets_mysql.csv`

> Phase 2 extends the period under analysis and adds operational variables, but it uses synthetic data. Its results demonstrate the analytical method and inform a possible pilot; they do not estimate real-world economic impact.

## 1. Load and Initial Inspection

The dataset is loaded and its dimensions, columns, data types, and first records are reviewed.

In [1]:
import pandas as pd

input_path = "phase2_support_tickets_raw.csv"
df = pd.read_csv(input_path)

print("Dimensions:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.info()
display(df.head())

Dimensions: (60000, 23)

Columns:
['ticket_id', 'customer_id', 'customer_age', 'customer_gender', 'product_purchased', 'ticket_created_at', 'first_response_at', 'resolved_at', 'ticket_type', 'ticket_subject', 'ticket_description', 'ticket_status', 'resolution', 'ticket_priority', 'priority_rule', 'ticket_channel', 'assigned_team', 'assigned_agent_id', 'escalated_flag', 'escalation_level', 'resolved_by_level', 'agent_handling_minutes', 'customer_satisfaction_rating']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 23 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ticket_id                     60000 non-null  int64  
 1   customer_id                   60000 non-null  object 
 2   customer_age                  60000 non-null  int64  
 3   customer_gender               60000 non-null  object 
 4   product_purchased             60000 non-null  object 
 5   ti

,ticket_id,customer_id,customer_age,customer_gender,product_purchased,ticket_created_at,first_response_at,resolved_at,ticket_type,ticket_subject,...,ticket_priority,priority_rule,ticket_channel,assigned_team,assigned_agent_id,escalated_flag,escalation_level,resolved_by_level,agent_handling_minutes,customer_satisfaction_rating
0,1,CUST009116,69,Male,Bose Headphones,2023-02-13 08:14:14,2023-02-13 16:38:14,2023-02-14 17:24:14,Refund request,Refund request,...,Low,"Low impact question, guidance request, or non-...",Email,Billing Support,AG070,0,NaN,L1,8,4.0
1,2,CUST011851,54,Female,GoPro Action Camera,2023-02-23 22:58:27,2023-02-24 02:35:27,2023-02-24 14:47:27,Cancellation request,Hardware issue,...,Medium,Standard issue affecting product usage with av...,Email,Technical Support L2,AG012,0,NaN,L1,15,5.0
2,3,CUST007197,61,Female,LG Washing Machine,2023-01-16 11:51:27,2023-01-16 12:29:27,2023-01-17 18:12:27,Refund request,Battery life,...,High,High customer impact or repeated failed troubl...,Chat,Technical Support L1,AG059,0,NaN,L1,12,3.0
3,4,CUST008082,65,Male,Dyson Vacuum Cleaner,2023-06-06 07:47:07,2023-06-06 08:05:07,NaN,Product inquiry,Hardware issue,...,Critical,"Critical impact, security/data risk, outage, o...",Email,Technical Support L2,AG073,0,NaN,Open,14,NaN
4,5,CUST003656,36,Female,Microsoft Surface,2023-03-19 13:20:28,2023-03-19 13:52:28,2023-03-19 18:34:28,Refund request,Payment issue,...,Critical,"Critical impact, security/data risk, outage, o...",Chat,Billing Support,AG064,0,NaN,L1,45,2.0


### Result

The dataset contains 60,000 tickets and 23 variables. Column names already use `snake_case`, but the three temporal variables were imported as text and must be converted to datetime values.

## 2. Identifier and Duplicate Validation

The analysis confirms that each row represents one ticket and that `ticket_id` contains neither null nor repeated values.

In [2]:
duplicated_rows = df.duplicated().sum()
duplicated_ids = df["ticket_id"].duplicated().sum()
null_ids = df["ticket_id"].isna().sum()

print("Fully duplicated rows:", duplicated_rows)
print("Duplicated ticket IDs:", duplicated_ids)
print("Null ticket IDs:", null_ids)

Fully duplicated rows: 0
Duplicated ticket IDs: 0
Null ticket IDs: 0


### Result

No duplicated rows, null identifiers, or repeated identifiers are found. `ticket_id` uniquely identifies each ticket.

## 3. Missing-Value Analysis

Columns containing missing values are identified, and the analysis determines whether the nulls are consistent with ticket status, resolution, and escalation.

In [3]:
nulls_by_column = df.isna().sum()

nulls_by_column = nulls_by_column[
    nulls_by_column > 0
].sort_values(ascending=False)

display(nulls_by_column.to_frame(name="null_count"))

,null_count
escalation_level,52014
resolved_at,10941
resolution,10941
customer_satisfaction_rating,10941
first_response_at,1117


### 3.1 Escalation Consistency

The escalation flag is compared with escalation level and ticket status.

In [4]:
df["escalated_flag"].value_counts(dropna=False)

escalated_flag
0    52014
1     7986
Name: count, dtype: int64

In [5]:
escalation_level_is_null = df["escalation_level"].isna()

escalation_table = pd.crosstab(
    df["escalated_flag"],
    escalation_level_is_null,
    margins=True
)

display(escalation_table)

escalation_level,False,True,All
escalated_flag,,,
0,0,52014,52014
1,7986,0,7986
All,7986,52014,60000


In [6]:
status_escalation_table = pd.crosstab(
    df["escalated_flag"],
    df["ticket_status"],
    margins=True
)

display(status_escalation_table)

ticket_status,Closed,Open,Pending Customer Response,All
escalated_flag,,,,
0,42505,5505,4004,52014
1,6554,829,603,7986
All,49059,6334,4607,60000


#### Result

The 52,014 non-escalated tickets have no escalation level, while all 7,986 escalated tickets do. No contradictions are found between `escalated_flag` and `escalation_level`.

Of the escalated tickets, 6,554 are closed and 1,432 remain open or pending.

### 3.2 Resolution-Data Consistency

The analysis checks whether missing resolution timestamps, resolution text, and satisfaction scores are explained by ticket status.

In [7]:
resolution_time_is_null = df["resolved_at"].isna()

status_resolution_table = pd.crosstab(
    df["ticket_status"],
    resolution_time_is_null,
    margins=True
)

display(status_resolution_table)

resolved_at,False,True,All
ticket_status,,,
Closed,49059,0,49059
Open,0,6334,6334
Pending Customer Response,0,4607,4607
All,49059,10941,60000


In [8]:
resolution_columns = [
    "resolution",
    "customer_satisfaction_rating"
]

nulls_by_status = (
    df.groupby("ticket_status")[resolution_columns]
    .agg(lambda column: column.isna().sum())
)

display(nulls_by_status)

,resolution,customer_satisfaction_rating
ticket_status,,
Closed,0,0
Open,6334,6334
Pending Customer Response,4607,4607


#### Result

All 49,059 closed tickets contain a resolution timestamp, resolution text, and satisfaction rating.

The 10,941 null values in these fields correspond exactly to open or pending tickets. They are structural nulls and do not require imputation.

### 3.3 First-Response Consistency

The analysis checks whether tickets without a first-response timestamp are cases that remain open.

In [9]:
first_response_is_null = df["first_response_at"].isna()

first_response_table = pd.crosstab(
    df["ticket_status"],
    first_response_is_null,
    margins=True
)

display(first_response_table)

first_response_at,False,True,All
ticket_status,,,
Closed,49059,0,49059
Open,5217,1117,6334
Pending Customer Response,4607,0,4607
All,58883,1117,60000


#### Result

There are 1,117 tickets without a recorded first response, and all of them remain open. These nulls are consistent with ticket status and are retained.

## 4. Date Conversion and Validation

Date columns are converted to `datetime`, and the process checks that conversion does not introduce new null values.

In [10]:
date_columns = [
    "ticket_created_at",
    "first_response_at",
    "resolved_at"
]

nulls_before = df[date_columns].isna().sum()

df[date_columns] = df[date_columns].apply(
    pd.to_datetime,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

nulls_after = df[date_columns].isna().sum()

date_validation = pd.DataFrame({
    "nulls_before": nulls_before,
    "nulls_after": nulls_after,
    "new_nulls_from_conversion": (
        nulls_after - nulls_before
    )
})

display(date_validation)
print(df[date_columns].dtypes)

,nulls_before,nulls_after,new_nulls_from_conversion
ticket_created_at,0,0,0
first_response_at,1117,1117,0
resolved_at,10941,10941,0


ticket_created_at    datetime64[ns]
first_response_at    datetime64[ns]
resolved_at          datetime64[ns]
dtype: object


### Result

All three temporal variables are converted successfully to `datetime`. Existing nulls are preserved and no new missing values are introduced.

### 4.1 Temporal Consistency

The analysis verifies that first responses and resolutions do not precede ticket creation and that resolutions do not precede the first response.

In [11]:
responses_before_creation = (
    df["first_response_at"].notna()
    & (
        df["first_response_at"]
        < df["ticket_created_at"]
    )
).sum()

resolutions_before_creation = (
    df["resolved_at"].notna()
    & (
        df["resolved_at"]
        < df["ticket_created_at"]
    )
).sum()

resolutions_before_first_response = (
    df["resolved_at"].notna()
    & df["first_response_at"].notna()
    & (
        df["resolved_at"]
        < df["first_response_at"]
    )
).sum()

print("First responses before ticket creation:", responses_before_creation)
print("Resolutions before ticket creation:", resolutions_before_creation)
print("Resolutions before the first response:", resolutions_before_first_response)

First responses before ticket creation: 0
Resolutions before ticket creation: 0
Resolutions before the first response: 0


#### Result

No first responses precede ticket creation, no resolutions precede ticket creation, and no resolutions precede the first response.

The temporal sequence is consistent across all available records.

## 5. Numeric Variable Validation

Customer age, agent handling time, and satisfaction are reviewed to detect impossible values or values outside expected ranges.

In [12]:
numeric_columns = [
    "customer_age",
    "agent_handling_minutes",
    "customer_satisfaction_rating"
]

numeric_summary = (
    df[numeric_columns]
    .describe()
    .T
)

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max
customer_age,60000.0,46.518583,16.748807,18.0,32.0,46.0,61.0,75.0
agent_handling_minutes,60000.0,28.216500,21.295985,2.0,14.0,23.0,37.0,240.0
customer_satisfaction_rating,49059.0,3.405124,0.941706,1.0,3.0,3.0,4.0,5.0


### 5.1 Numeric-Range Validation

Values that violate the minimum quality rules are counted explicitly:

- age outside the 0-120 range;
- negative agent handling time;
- satisfaction outside the 1-5 scale;
- non-integer satisfaction values.

In [13]:
rating_is_present = df[
    "customer_satisfaction_rating"
].notna()

numeric_range_validation = pd.Series({
    "ages_outside_0_120": (
        ~df["customer_age"].between(0, 120)
    ).sum(),

    "negative_handling_times": (
        df["agent_handling_minutes"] < 0
    ).sum(),

    "ratings_outside_1_5": (
        rating_is_present
        & ~df["customer_satisfaction_rating"].between(1, 5)
    ).sum(),

    "non_integer_ratings": (
        rating_is_present
        & (
            df["customer_satisfaction_rating"] % 1 != 0
        )
    ).sum()
})

numeric_range_validation

ages_outside_0_120           0
negative_handling_times    0
ratings_outside_1_5            0
non_integer_ratings              0
dtype: int64

#### Result

No ages fall outside the defined range, no handling times are negative, and no satisfaction ratings fall outside the 1-5 scale. Populated ratings use integer values.

In [14]:
tickets_without_first_response = df[
    df["first_response_at"].isna()
]

tickets_without_first_response[
    ["ticket_status", "first_response_at", "agent_handling_minutes"]
].describe(include="all")

,ticket_status,first_response_at,agent_handling_minutes
count,1117,0,1117.000000
unique,1,NaN,NaN
top,Open,NaN,NaN
freq,1117,NaN,NaN
mean,NaN,NaT,10.324978
min,NaN,NaT,2.000000
25%,NaN,NaT,5.000000
50%,NaN,NaT,8.000000
75%,NaN,NaT,14.000000
max,NaN,NaT,64.000000


### Result

Customer ages range from 18 to 75, satisfaction ratings from 1 to 5, and agent handling time from 2 to 240 minutes.

The 1,117 tickets without a first response are open and contain between 2 and 64 minutes of handling time. This may represent work started before an agent sends a response, so the values are not modified without a supporting business rule.

The maximum handling time of 240 minutes is high, but it is not treated as erroneous because no business threshold invalidates it.

## 6. MySQL Preparation

Maximum lengths are calculated for text variables to support appropriate MySQL data types.

In [15]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

length_records = []

for column in text_columns:
    values = df[column].dropna().astype(str)

    maximum_length = (
        int(values.str.len().max())
        if not values.empty
        else 0
    )

    length_records.append({
        "column": column,
        "maximum_length": maximum_length
    })

text_lengths = pd.DataFrame(length_records)

display(text_lengths)

,column,maximum_length
0,customer_id,10
1,customer_gender,6
2,product_purchased,30
3,ticket_type,20
4,ticket_subject,24
5,ticket_description,151
6,ticket_status,25
7,resolution,94
8,ticket_priority,8
9,priority_rule,65


### Result

Observed maximum lengths are used as a reference when defining MySQL text types and preventing truncation during import.

## 7. Export and Final Validation

The transformed dataset is exported with a consistent datetime format, UTF-8 encoding, and `\N` as the representation of MySQL null values.

In [16]:
mysql_output_file = "phase2_support_tickets_mysql.csv"

df.to_csv(
    mysql_output_file,
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d %H:%M:%S",
    na_rep="\\N",
    lineterminator="\n"
)

print("File created:", mysql_output_file)
print("Dimensions:", df.shape)

File created: phase2_support_tickets_mysql.csv
Dimensions: (60000, 23)


In [17]:
df_export_check = pd.read_csv(
    mysql_output_file,
    na_values=[r"\N"],
    keep_default_na=False
)

export_validation = pd.Series({
    "rows_before": len(df),
    "rows_after": len(df_export_check),
    "same_row_count": len(df) == len(df_export_check),
    "same_column_count": len(df.columns) == len(df_export_check.columns),
    "same_column_names_and_order": (
        df.columns.tolist() == df_export_check.columns.tolist()
    ),
    "same_ticket_ids_and_order": (
        df["ticket_id"].astype(str).tolist()
        == df_export_check["ticket_id"].astype(str).tolist()
    ),
    "nulls_before": int(df.isna().sum().sum()),
    "nulls_after": int(df_export_check.isna().sum().sum())
})

export_validation

rows_before                           60000
rows_after                         60000
same_row_count                  True
same_column_count               True
same_column_names_and_order     True
same_ticket_ids_and_order               True
nulls_before                           85954
nulls_after                         85954
dtype: object

### Result

The exported file preserves all 60,000 rows, 23 columns, column order, ticket identifiers, and null values from the transformed dataset.

The dataset is ready for loading into MySQL, with no structural loss detected during export.